# DSWP manuscript figure components

This notebook assembles only panels supported by the released derived tables. Figures are returned in memory and are not saved or embedded.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from dswp.io import load_buzz_events
from dswp.classification import attach_labels
from dswp.manuscript_figures import dswp_component_figure, dswp_model_selection_figure
EVENTS = ROOT / 'data/dswp/buzz_events.csv'
ROLL = ROOT / 'data/dswp/roll_speed_bins.csv'
STRAIGHTNESS = ROOT / 'results/dswp/buzz_straightness.csv'
ENERGY = ROOT / 'results/dswp/individual_energy.csv'
TRANSITIONS = ROOT / 'results/dswp/buzz_markov_transitions.csv'


Known issue touchpoint: classifier speed variable. The registered classifier uses `avg_speed_3d_m_s`; pitch-adaptive speed is retained as a separate sensitivity.

Known issue touchpoint: diagnostic duration. The model-selection diagnostic uses ramp plus plateau duration, not full first-click--last-click duration.

Known issue touchpoint: Markov resampling. The figure preserves the result-producing deployment-level leave-subset-out implementation.

Known issue touchpoint: terminal buzzes. Pause panels use the released pause field and do not infer a new terminal-event rule.

Known issue touchpoint: recurrence choice. Comeback alternatives are audited in the companion notebook and are not selected here.


In [ ]:
events = attach_labels(load_buzz_events(EVENTS), method='gmm')
if STRAIGHTNESS.exists():
    events = events.merge(pd.read_csv(STRAIGHTNESS), on='event_id', how='left', validate='one_to_one')
roll_bins = pd.read_csv(ROLL) if ROLL.exists() else None
energy = pd.read_csv(ENERGY) if ENERGY.exists() else None
transitions = pd.read_csv(TRANSITIONS) if TRANSITIONS.exists() else None
events[['buzz_type']].value_counts()


In [ ]:
figure, availability = dswp_component_figure(events, roll_bins=roll_bins, energy=energy, transitions=transitions)
availability


In [ ]:
model_figure, model_tables = dswp_model_selection_figure(events)
model_tables['agreement'].round(3)


The assembled figure covers classifier, individual proportions, depth, geometry residuals, straightness, pause, roll bins, energy, Markov, and classifier robustness. GMM/BIC and all available components are routed through tested analytical modules.

The unavailable panels are deliberate: the full-record dive trace and location-bearing regional map/vessel track require inputs outside the safe release. Buzz construction is reproduced in notebook 00 and pooled trajectory analyses in the trajectory notebooks.
